# Mission d'adaptation : un nouveau site, sans code fourni
**ÉNONCÉ**

Vous savez maintenant scraper et crawler un site (Books to Scrape), et le faire poliment. **Cette mission change de site, et on ne vous donne aucun code de collecte.** C'est à vous d'adapter votre démarche à une structure différente.

**Le site :** `quotes.toscrape.com`, des citations célèbres, créé lui aussi pour s'entraîner.

## Ce que vous devez produire
1. **`citations`** : un tableau de **toutes** les citations (toutes les pages), avec les colonnes `texte`, `auteur`, `tags`, `url_page`, `url_auteur`.
2. **`auteurs`** : un tableau de la fiche de **chaque auteur, une seule fois**, avec `url_auteur`, `auteur`, `date_naissance`, `lieu_naissance`, `biographie`.
3. **`nb_requetes`** : le nombre de pages que votre script a demandées (pages de citations + fiches d'auteurs, sans compter `robots.txt`).
4. **`citations_completes`** : les citations enrichies de la fiche de leur auteur.
5. **`top_auteurs`** et **`top_tags`** : les 5 auteurs les plus cités et les 10 tags les plus fréquents.
6. **Une note de 5 lignes** : ce qui change par rapport à Books to Scrape, et comment vous l'avez géré.

## Règles
Les règles du notebook précédent s'appliquent : vous vous **présentez** (User-Agent avec contact), vous **lisez le robots.txt**, vous faites une **pause**. Ces outils vous sont fournis plus bas.

## Pour vous mettre sur la piste (sans code)
- **Inspectez** le site dans votre navigateur avant d'écrire quoi que ce soit. Qu'est-ce qui est pareil ? Qu'est-ce qui est différent ?
- Regardez **la forme des adresses** des liens « Next » et « (about) ».
- Un **même auteur** apparaît dans de nombreuses citations. Faut-il télécharger sa fiche à chaque fois ?
- Une citation a **plusieurs tags**. Comment stocker cela dans une seule case ?
- Le texte d'une citation est entouré de **guillemets typographiques**. Faut-il les garder ?

In [4]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"      # un NOUVEAU site-école, à la structure différente de Books to Scrape

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Se présenter
Comme dans le notebook précédent : remplacez `___` par votre contact (adresse e-mail ou adresse d'une page de projet).

In [5]:
ENTETES = {"User-Agent": "TPFormation/1.0 (greetaiates@gmail.com) python-requests"}
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (greetaiates@gmail.com) python-requests'}


In [6]:
# Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact')

✅ User-Agent renseigné : TPFormation/1.0 (greetaiates@gmail.com) python-requests


## Étape 2 — Vos outils (fournis)
La fonction de requête polie et la lecture du `robots.txt`, issues du notebook précédent.

In [7]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)
        reponse = requests.get(url, headers=ENTETES, timeout=20)
        if reponse.status_code in (429, 503):
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser(); regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 3 — Consulter le robots.txt
Lisez le `robots.txt` de ce site, puis demandez s'il est permis de visiter la page 2 des citations (`/page/2/`) et la fiche d'un auteur (`/author/Albert-Einstein`). Stockez les deux réponses (vrai ou faux) dans `autorise_pages` et `autorise_auteur`.

In [8]:
regles = lire_robots(SITE)
# TODO : calculer autorise_pages et autorise_auteur (si regles vaut None, aucune restriction annoncée)

AGENT = ENTETES.get('User-Agent', '*')

if regles is None:
    autorise_pages = True
    autorise_auteur = True
else:
    autorise_pages = regles.can_fetch(AGENT, LIVRES + '/')
    autorise_auteur = regles.can_fetch(AGENT, SITE + '/author/')


https://quotes.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée


In [9]:
# Contrôle (ne pas modifier)
controle(isinstance(autorise_pages, bool) and isinstance(autorise_auteur, bool), 'Autorisations établies : pages=%s, auteur=%s' % (autorise_pages, autorise_auteur), 'Attendu : deux booléens')

✅ Autorisations établies : pages=True, auteur=True


## Étape 4 — Collecter toutes les citations *(à écrire seul)*
Objectif : la variable **`citations`** (un `DataFrame`) et le compteur **`nb_requetes`**. Cette cellule est **vide** : à vous d'écrire la boucle.

In [10]:
# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)
from urllib.parse import urljoin

# À vous : collecter toutes les citations (variables attendues : citations, nb_requetes)
nb_requetes = 0
lignes = []
url = SITE + '/'          # page de départ (à adapter si le notebook donne une autre constante)

while url:
    r = get_poli(url)
    nb_requetes += 1
    r.encoding = "utf-8"
    s = BeautifulSoup(r.text, "html.parser")

    for q in s.select("div.quote"):
        lien_auteur = q.select_one("a[href^='/author']")
        lignes.append({
            "texte": q.select_one("span.text").text.strip().strip("“”\""),
            "auteur": q.select_one("small.author").text.strip(),
            "tags": [t.text for t in q.select("a.tag")],        # une vraie liste
            "url_page": url,
            "url_auteur": urljoin(url, lien_auteur["href"]),    # lien relatif -> absolu
        })

    suivant = s.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None    # s'arrête à la dernière page

citations = pd.DataFrame(lignes).drop_duplicates(subset=["texte", "auteur"]).reset_index(drop=True)
print(nb_requetes, "requêtes,", len(citations), "citations")

10 requêtes, 100 citations


In [11]:
# Contrôle (ne pas modifier)
colonnes = {"texte", "auteur", "tags", "url_page", "url_auteur"}
controle(isinstance(citations, pd.DataFrame) and colonnes <= set(citations.columns) and len(citations) >= 10,
         "%d citations collectées, colonnes correctes" % len(citations), "Attendu : un DataFrame avec les colonnes " + ", ".join(sorted(colonnes)) + " et au moins 10 lignes")
controle(citations["tags"].map(lambda t: isinstance(t, list)).all() and not citations["texte"].str.startswith(("“", '"')).any(),
         "Les tags sont des listes et les guillemets typographiques sont retirés", "Les tags doivent être des listes, et le texte ne doit plus commencer par un guillemet")
controle(citations["url_auteur"].str.startswith("http").all() and citations["url_page"].str.startswith("http").all(),
         "Toutes les adresses sont complètes", "Les adresses doivent commencer par http : pensez aux liens relatifs")
controle(not citations.duplicated(subset=["texte", "auteur"]).any(), "Aucune citation en double", "Il y a des citations en double : une page a-t-elle été lue deux fois ?")

✅ 100 citations collectées, colonnes correctes
✅ Les tags sont des listes et les guillemets typographiques sont retirés
✅ Toutes les adresses sont complètes
✅ Aucune citation en double


## Étape 5 — Collecter la fiche de chaque auteur *(à écrire seul)*
Objectif : la variable **`auteurs`**, avec **une seule ligne par auteur**. Inspectez une fiche d'auteur pour trouver où sont la date et le lieu de naissance, et la biographie. **Attention au nombre de requêtes** : le contrôle suivant le vérifie.

In [12]:
# À vous : collecter une fiche par auteur (variable attendue : auteurs)
# Étape 5 : une requête par auteur distinct, pas une par citation
urls_auteurs = citations["url_auteur"].drop_duplicates().tolist()
print(len(urls_auteurs), "auteurs distincts")

lignes_auteurs = []
for u in urls_auteurs:
    r = get_poli(u)
    nb_requetes += 1
    r.encoding = "utf-8"
    s = BeautifulSoup(r.text, "html.parser")

    lieu = s.select_one("span.author-born-location").text.strip()
    if lieu.lower().startswith("in "):
        lieu = lieu[3:]                      # retire le « in » initial

    lignes_auteurs.append({
        "auteur": s.select_one("h3.author-title").text.strip(),
        "date_naissance": s.select_one("span.author-born-date").text.strip(),
        "lieu_naissance": lieu,
        "biographie": s.select_one("div.author-description").text.strip(),
        "url_auteur": u,
    })

auteurs = pd.DataFrame(lignes_auteurs).drop_duplicates(subset=["url_auteur"]).reset_index(drop=True)
print(len(auteurs), "fiches,", nb_requetes, "requêtes au total")

50 auteurs distincts
50 fiches, 60 requêtes au total


In [13]:
# Contrôle (ne pas modifier)
colonnes_a = {"url_auteur", "auteur", "date_naissance", "lieu_naissance", "biographie"}
controle(isinstance(auteurs, pd.DataFrame) and colonnes_a <= set(auteurs.columns) and auteurs["url_auteur"].is_unique,
         "%d fiches d'auteurs, une seule par auteur" % len(auteurs), "Attendu : les colonnes " + ", ".join(sorted(colonnes_a)) + " et une seule ligne par auteur")
controle(set(citations["url_auteur"]) <= set(auteurs["url_auteur"]), "Tous les auteurs cités ont une fiche", "Certains auteurs cités n'ont pas de fiche")
attendu = citations["url_page"].nunique() + citations["url_auteur"].nunique()
controle(nb_requetes <= attendu + 2, "%d requêtes pour %d attendues au minimum : fiches non redemandées" % (nb_requetes, attendu), "%d requêtes alors que %d suffisent : un auteur a été redemandé plusieurs fois" % (nb_requetes, attendu))

✅ 50 fiches d'auteurs, une seule par auteur
✅ Tous les auteurs cités ont une fiche
✅ 60 requêtes pour 60 attendues au minimum : fiches non redemandées


## Étape 6 — Fusionner et analyser *(à écrire seul)*
Objectifs : **`citations_completes`** (chaque citation avec la fiche de son auteur), **`top_auteurs`** (les 5 auteurs les plus cités) et **`top_tags`** (les 10 tags les plus fréquents). Indice : pour compter des tags stockés dans des listes, il faut d'abord « déplier » ces listes.

In [14]:
# À vous : citations_completes, top_auteurs, top_tags

citations_completes = citations.merge(
    auteurs.drop(columns="auteur"),   # évite les colonnes auteur_x / auteur_y
    on="url_auteur",
    how="left",                       # garde toutes les citations
)

# 5 auteurs les plus cités
top_auteurs = citations["auteur"].value_counts().head(5)

# 10 tags les plus fréquents : on déplie d'abord les listes
top_tags = citations.explode("tags")["tags"].value_counts().head(10)

print(top_auteurs, "\n")
print(top_tags)


auteur
Albert Einstein    10
J.K. Rowling        9
Marilyn Monroe      7
Dr. Seuss           6
Mark Twain          6
Name: count, dtype: int64 

tags
love             14
inspirational    13
life             13
humor            12
books            11
reading           7
friendship        5
friends           4
truth             4
simile            3
Name: count, dtype: int64


In [15]:
# Contrôle (ne pas modifier)
controle(len(citations_completes) == len(citations) and "date_naissance" in citations_completes.columns and citations_completes["date_naissance"].notna().all(),
         "Fusion correcte : chaque citation a la fiche de son auteur", "La fusion doit garder toutes les citations, chacune avec la date de naissance de son auteur")
controle(len(top_auteurs) == min(5, citations["auteur"].nunique()) and len(top_tags) == min(10, citations.explode("tags")["tags"].nunique()) and top_auteurs.is_monotonic_decreasing and top_tags.is_monotonic_decreasing,
         "Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent", "Attendu : 5 auteurs et 10 tags triés par fréquence décroissante")

✅ Fusion correcte : chaque citation a la fiche de son auteur
✅ Classements corrects : 5 auteurs et 10 tags, du plus fréquent au moins fréquent


## Étape 7 — Votre note (5 lignes)
Répondez ici, en 5 lignes maximum :

1. Ce qui était **identique** à Books to Scrape.
2. Ce qui était **différent**.
3. Ce que vous avez dû **changer** dans votre démarche.
4. Comment vous avez évité de **redemander** les fiches d'auteurs.
5. Ce qui, à votre avis, **casserait** votre script si le site changeait.

> 

## Réponse

1. **Identique** : la démarche est la même : `requests` pour télécharger, BeautifulSoup pour lire, une boucle qui suit le bouton « next » jusqu'à ce qu'il disparaisse, des requêtes polies avec `get_poli` (pause, `User-Agent` avec contact), et `urljoin` pour les liens relatifs.

2. **Différent** : les sélecteurs changent (`div.quote`, `span.text`, `small.author`), il n'y a pas de note en étoiles, chaque citation a plusieurs tags, le texte est entouré de guillemets typographiques, et les auteurs ont leur propre fiche. Le site n'a pas de `robots.txt` (code 404), donc aucune restriction annoncée.

3. **Changé** : j'ai stocké les tags dans une liste Python plutôt qu'une chaîne, retiré les guillemets avec `strip`, et ajouté une seconde collecte, celle des fiches d'auteurs, à partir des liens « (about) » relevés dans les citations.

4. **Fiches non redemandées** : je ne boucle pas sur les 100 citations mais sur `citations["url_auteur"].drop_duplicates()`, soit 50 fiches, ce qui donne 60 requêtes (10 pages et 50 fiches) au lieu de 110.

5. **Fragilité** : le renommage d'une classe CSS (`div.quote`, `li.next`, `span.author-born-date`…) ou un changement de structure des liens. Un `select_one` renvoie alors `None` et le script plante avec une erreur, ou, pire, la boucle s'arrête sans erreur si `li.next` disparaît.